In [1]:
import pandas as pd
import numpy as np
import itertools
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import PassiveAggressiveClassifier,LogisticRegression
from sklearn.naive_bayes import MultinomialNB,GaussianNB,ComplementNB,BernoulliNB,CategoricalNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn import metrics
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt

In [2]:
dataset=pd.read_csv("whatsapp_chat_labeled.csv")
dataset

,Date,Time,Name,Chat,Label
0,10/14/26,1:47 PM,Nivetha,Good morning,Normal
1,10/14/26,1:47 PM,Nivetha,How are you?,Normal
2,10/14/26,1:49 PM,Kirthi,Hi,Normal
3,10/14/26,1:50 PM,Kirthi,Good..u?,Normal
4,10/14/26,1:50 PM,Nivetha,Fine😊,Normal
...,...,...,...,...,...
144,10/19/26,12:45 PM,Kirthi,"Sure, come over and I'll give you an empty bowl😂",Normal
145,10/19/26,12:45 PM,Nivetha,"Wow, such hospitality! I deserve better from you.",Flirty
146,10/19/26,12:46 PM,Kirthi,May be you do. I'll consider it😜,Flirty
147,10/19/26,12:46 PM,Nivetha,"I'll remember that. Anyway, I need to get back...",Normal


In [5]:
x=dataset["Chat"]
y=dataset["Label"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.33, random_state=53)

In [9]:
hash_vectorizer=HashingVectorizer(n_features=10, norm=None, alternate_sign=False,stop_words='english')
hash_train = hash_vectorizer.fit_transform(X_train)
hash_test = hash_vectorizer.transform(X_test)
print(hash_train.toarray())

[[1. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 1. 0.]
 [0. 0. 0. 0. 2. 0. 2. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 1.]
 [1. 1. 0. 0. 0. 1. 0. 1. 0. 0.]
 [1. 1. 0. 0. 1. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 2. 0. 1.]
 [0. 0. 0. 0. 0. 1. 0. 1. 0. 1.]
 [0. 1. 0. 0. 0. 2. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 1. 0. 0.]
 [0. 1. 0. 0. 1. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 1.]
 [0. 1. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 1. 1. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [3. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 2. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 1. 0. 0. 0. 1. 0. 0.]
 [0. 1. 1. 0. 0. 0. 0. 0. 1. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [1. 0. 0. 1. 1. 1. 0. 0. 1. 0.]
 [1. 0. 0. 1. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0.

In [23]:
param_grid = {'alpha':[0.1,0.01,1.0,5.0]}
grid = GridSearchCV(MultinomialNB(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(hash_train, y_train) 
grid_predictions = grid.predict(hash_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 4 candidates, totalling 12 fits
The f1_macro value for best parameter {'alpha': 0.01}: 0.5383380168606985
The confusion Matrix:
 [[12 18]
 [ 5 15]]
The report:
               precision    recall  f1-score   support

      Flirty       0.71      0.40      0.51        30
      Normal       0.45      0.75      0.57        20

    accuracy                           0.54        50
   macro avg       0.58      0.57      0.54        50
weighted avg       0.61      0.54      0.53        50



In [12]:
param_grid = {'C':[0.1,0.01,1.0],'loss':["hinge","squared_hinge"]}
grid = GridSearchCV(PassiveAggressiveClassifier(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(hash_train, y_train) 
grid_predictions = grid.predict(hash_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 6 candidates, totalling 18 fits
The f1_macro value for best parameter {'C': 0.1, 'loss': 'hinge'}: 0.5716034271725826
The confusion Matrix:
 [[18 12]
 [ 9 11]]
The report:
               precision    recall  f1-score   support

      Flirty       0.67      0.60      0.63        30
      Normal       0.48      0.55      0.51        20

    accuracy                           0.58        50
   macro avg       0.57      0.57      0.57        50
weighted avg       0.59      0.58      0.58        50



In [13]:
param_grid = {'C':[0.1,0.01,1.0],'penalty':['l2'],'solver':['lbfgs','liblinear','newton-cg','newton-cholesky', 'sag', 'saga']}
grid = GridSearchCV(LogisticRegression(random_state=0,max_iter=1000), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(hash_train, y_train) 
grid_predictions = grid.predict(hash_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 18 candidates, totalling 54 fits
The f1_macro value for best parameter {'C': 1.0, 'penalty': 'l2', 'solver': 'lbfgs'}: 0.5398159263705482
The confusion Matrix:
 [[13 17]
 [ 6 14]]
The report:
               precision    recall  f1-score   support

      Flirty       0.68      0.43      0.53        30
      Normal       0.45      0.70      0.55        20

    accuracy                           0.54        50
   macro avg       0.57      0.57      0.54        50
weighted avg       0.59      0.54      0.54        50



In [14]:
param_grid = {'C':[0.1,0.01,1.0],'kernel':['linear','poly','rbf','sigmoid'],'gamma':['scale','auto']}
grid = GridSearchCV(SVC(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(hash_train, y_train) 
grid_predictions = grid.predict(hash_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 24 candidates, totalling 72 fits
The f1_macro value for best parameter {'C': 1.0, 'gamma': 'scale', 'kernel': 'linear'}: 0.5571658615136876
The confusion Matrix:
 [[12 18]
 [ 4 16]]
The report:
               precision    recall  f1-score   support

      Flirty       0.75      0.40      0.52        30
      Normal       0.47      0.80      0.59        20

    accuracy                           0.56        50
   macro avg       0.61      0.60      0.56        50
weighted avg       0.64      0.56      0.55        50



In [19]:
param_grid = {'n_estimators':[10,50,100],'criterion':['gini','entropy','log_loss'],'min_samples_split':[2,3,4],'max_features':['sqrt','log2'],'class_weight':['balanced','balanced_subsample']}
grid = GridSearchCV(RandomForestClassifier(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(hash_train, y_train) 
grid_predictions = grid.predict(hash_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 108 candidates, totalling 324 fits
The f1_macro value for best parameter {'class_weight': 'balanced', 'criterion': 'log_loss', 'max_features': 'sqrt', 'min_samples_split': 4, 'n_estimators': 10}: 0.6124031007751938
The confusion Matrix:
 [[19 11]
 [ 8 12]]
The report:
               precision    recall  f1-score   support

      Flirty       0.70      0.63      0.67        30
      Normal       0.52      0.60      0.56        20

    accuracy                           0.62        50
   macro avg       0.61      0.62      0.61        50
weighted avg       0.63      0.62      0.62        50



In [20]:
param_grid = {'criterion':['gini','entropy','log_loss'],'splitter':['best','random'],'min_samples_split':[2,3,4],'max_features':['sqrt','log2'],'class_weight':['balanced']}
grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(hash_train, y_train) 
grid_predictions = grid.predict(hash_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 36 candidates, totalling 108 fits
The f1_macro value for best parameter {'class_weight': 'balanced', 'criterion': 'gini', 'max_features': 'sqrt', 'min_samples_split': 3, 'splitter': 'random'}: 0.6486151302190988
The confusion Matrix:
 [[21  9]
 [ 8 12]]
The report:
               precision    recall  f1-score   support

      Flirty       0.72      0.70      0.71        30
      Normal       0.57      0.60      0.59        20

    accuracy                           0.66        50
   macro avg       0.65      0.65      0.65        50
weighted avg       0.66      0.66      0.66        50



In [25]:
param_grid = {'n_neighbors':[2,3,4,5],'weights':['uniform','distance'],'algorithm':['auto','ball_tree','kd_tree','brute'],'p':[1,2]}
grid = GridSearchCV(KNeighborsClassifier(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(hash_train, y_train) 
grid_predictions = grid.predict(hash_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 64 candidates, totalling 192 fits
The f1_macro value for best parameter {'algorithm': 'auto', 'n_neighbors': 4, 'p': 2, 'weights': 'uniform'}: 0.5941558441558441
The confusion Matrix:
 [[18 12]
 [ 8 12]]
The report:
               precision    recall  f1-score   support

      Flirty       0.69      0.60      0.64        30
      Normal       0.50      0.60      0.55        20

    accuracy                           0.60        50
   macro avg       0.60      0.60      0.59        50
weighted avg       0.62      0.60      0.60        50



C:\Anaconda3\envs\aiml\Lib\site-packages\sklearn\model_selection\_search.py:1135: UserWarning: One or more of the test scores are non-finite: [       nan 0.63231803 0.61270934 0.59445921        nan 0.6227172
 0.58136123 0.57167935        nan 0.56672954 0.69142576 0.66590246
        nan 0.58897423 0.57657891 0.58494493        nan 0.63231803
 0.61270934 0.59445921        nan 0.6227172  0.58136123 0.57167935
        nan 0.56672954 0.69142576 0.66590246        nan 0.58897423
 0.57657891 0.58494493        nan 0.63231803 0.61270934 0.59445921
        nan 0.6227172  0.58136123 0.57167935        nan 0.56672954
 0.69142576 0.66590246        nan 0.58897423 0.57657891 0.58494493
        nan 0.63231803 0.61270934 0.59445921        nan 0.6227172
 0.58136123 0.57167935        nan 0.56672954 0.69142576 0.66590246
        nan 0.58897423 0.57657891 0.58494493]
  warnings.warn(
